# 08 — Paper figures

Final figures + tables for the paper / resume bullet:
  - Figure 1: cumulative OOS returns (paper-style)
  - Figure 2: Sharpe by realized-vol regime (paper-style)
  - Headline summary table (Sharpe / vol / max-DD per rung)
  - Decomposition table (F3 + F2 gaps)
  - Net-Sharpe-vs-TC curve


In [ ]:
import sys; from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src import config, eval as evalmod


## Figure 1 — cumulative P&L (vol-matched)


In [ ]:
# Replicate paper's Figure 1 with our 9 rungs (vs paper's 5)
RUNG_IDS = ['rung_1_simple_momentum_decile', 'rung_2_ewm_momentum_decile',
            'rung_3_ts_regression_decile', 'rung_4_linear_tcnn',
            'rung_5_tcnn_1ch', 'rung_6_tcnn_3ch']
master = evalmod.load_master_results(RUNG_IDS)

# Vol-match each rung to rung 1's volatility (paper convention)
target_vol = master[master['experiment_id'] == RUNG_IDS[0]]['return'].std() * np.sqrt(252)

fig, ax = plt.subplots(figsize=(14, 6))
for exp_id in RUNG_IDS:
    df = master[master['experiment_id'] == exp_id].sort_values('date')
    raw_vol = df['return'].std() * np.sqrt(252)
    scale = target_vol / (raw_vol + 1e-12)
    ax.plot(df['date'], (df['return'] * scale).cumsum(), label=exp_id, alpha=0.85)
ax.set(title=f'Cumulative OOS Returns (vol-matched to rung 1, vol={target_vol*100:.1f}%)',
       ylabel='cumulative return')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()


## Headline summary table


In [ ]:
summary = (master.groupby('experiment_id')['return']
                  .apply(lambda s: pd.Series(evalmod.perf_summary(s)))
                  .unstack()
                  .loc[RUNG_IDS, ['ann_return', 'ann_vol', 'sharpe', 'max_dd', 't_stat']]
                  .round(3))
summary
